# Evaluating and Debugging a Classification Pipeline

*Notebook #2.5 in the hands-on MNE series. Assumes the material of notebooks #1 (EEG basics, filtering, epoching), #1.5 (preprocessing decisions), and #2 (motor imagery, CSP, cross-validated classification).*

In notebook #2 you built a working motor-imagery classifier: CSP spatial filtering, LDA classification, cross-validated accuracy around 70–80%. The pipeline ran, the number came out, and it was above chance. Done?

Not quite. In practice, the hardest part of BCI research is not getting *an* accuracy number — it is knowing whether that number is **trustworthy**, whether it is **good enough**, and what to do when it is **not**. This notebook addresses three questions that every BCI practitioner must learn to answer:

1. *Is my accuracy statistically meaningful, or could it be a fluke?*
2. *Is my pipeline sound, or have I inadvertently leaked information from test into training data?*
3. *My accuracy is low — is the problem in my data, my preprocessing, my features, or my classifier?*

These are not theoretical concerns. Published BCI papers have been retracted due to data leakage. Clinical trials have been designed around accuracies that turned out to be inflated. The skills in this notebook protect you from these failures.

> **Pedagogical note.** Each scenario begins by presenting a realistic situation and asking you to diagnose the problem before seeing the answer. The goal is to develop the instinct for *where* to look when something is wrong.

## Table of contents

1. **Data preparation** — Loading the EEGBCI motor-imagery data and reproducing the baseline pipeline from notebook #2.
2. **Scenario A: The sceptic** — *"Is 72% accuracy real, or could chance produce this?"* → Permutation testing.
3. **Scenario B: The auditor** — *"Is my cross-validation actually valid?"* → Data leakage and how to detect it.
4. **Scenario C: The debugger** — *"Accuracy is low — where is the problem?"* → Systematic diagnosis through ablation.
5. **Scenario D: The realist** — *"This works for subject 1 — does it work for everyone?"* → Cross-subject variability and BCI illiteracy.
6. **Synthesis** — A diagnostic checklist for classification pipelines.
7. **Practice scenarios** — Unsolved problems for self-assessment.

## Position in the textbook

- **Rao Ch. 5 — Machine Learning.** Evaluation methodology, overfitting, and generalisation.
- **Rao Ch. 6 — Building a BCI.** Practical constraints that determine whether an offline accuracy translates to usable online performance.
- **Rao Ch. 10 — Current Research.** The BCI illiteracy problem and inter-subject variability.

## 1. Data preparation

We reproduce the motor-imagery pipeline from notebook #2 in compact form: load data for one subject, filter to the μ+β band, epoch around imagery cues, and build the CSP+LDA classifier. This is the baseline against which all scenarios below are evaluated.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import mne
from mne.datasets import eegbci
from mne.channels import make_standard_montage
from mne.io import concatenate_raws, read_raw_edf
from mne.decoding import CSP

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import (
    ShuffleSplit, StratifiedKFold, cross_val_score,
    cross_val_predict, learning_curve, permutation_test_score,
)
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

print("MNE:", mne.__version__)

In [ ]:
def load_mi_subject(subject=1, runs=[4, 8, 12]):
    """Load and preprocess MI data for one subject. Returns X, y, epochs."""
    raw_fnames = eegbci.load_data(subject, runs, update_path=True)
    raw = concatenate_raws([read_raw_edf(f, preload=True) for f in raw_fnames])
    eegbci.standardize(raw)
    raw.set_montage(make_standard_montage("standard_1005"))
    raw.filter(l_freq=7.0, h_freq=30.0)

    events, event_id = mne.events_from_annotations(raw)
    # Keep only left (T1=2) and right (T2=3) imagery events.
    picks = mne.pick_types(raw.info, eeg=True, stim=False)
    event_id_lr = {k: v for k, v in event_id.items() if v in (2, 3)}

    epochs = mne.Epochs(raw, events, event_id_lr,
                        tmin=0.5, tmax=3.5, baseline=None,
                        picks=picks, preload=True)

    X = epochs.get_data(copy=False) * 1e6
    y = epochs.events[:, -1]
    return X, y, epochs


X, y, epochs = load_mi_subject(subject=1)
print(f"X: {X.shape}  (trials × channels × times)")
print(f"y: {np.unique(y, return_counts=True)}")

In [ ]:
# Baseline pipeline from notebook #2.
clf = Pipeline([("CSP", CSP(n_components=4, reg=None, log=True, norm_trace=False)),
                ("LDA", LinearDiscriminantAnalysis())])

cv = ShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
scores_baseline = cross_val_score(clf, X, y, cv=cv)

print(f"Baseline accuracy: {scores_baseline.mean():.1%} ± {scores_baseline.std():.1%}")
print(f"Per-fold: {np.round(scores_baseline, 3)}")

The pipeline produces a number. The four scenarios below ask whether that number can be trusted, and what to do when it cannot.

---

---

## 2. Scenario A — The sceptic

### The situation

You report 72% accuracy to your supervisor. The response:

> *"Chance is 50%. You got 72%. But with only 45 trials per class, couldn't random fluctuations produce that? How do you know this accuracy is statistically significant?"*

This is a legitimate question. With small sample sizes — common in BCI research — even 70% accuracy may not be reliably above chance.

### ❓ Pause — your prediction

1. Is comparing your accuracy to 50% (theoretical chance) sufficient to claim significance? Why might this be misleading?
2. If you shuffled the labels randomly and re-ran the pipeline, what accuracy would you expect? Would it always be exactly 50%?
3. How many times would you need to repeat the shuffle-and-classify procedure to build a reliable null distribution?

---

### The permutation test

A permutation test answers the question: *if there were no true relationship between neural data and labels, how often would cross-validation produce an accuracy as high as the one I observed?*

The procedure:
1. Randomly shuffle the label vector `y`, breaking any real association between brain data and class.
2. Run the full cross-validation pipeline on the shuffled labels.
3. Record the resulting accuracy.
4. Repeat steps 1–3 many times (typically 100–1000) to build a **null distribution** of accuracies.
5. Compute the p-value: the fraction of permutation accuracies that equal or exceed the observed accuracy.

If p < 0.05, the observed accuracy is unlikely to have arisen by chance.

In [ ]:
# scikit-learn provides this directly.
# n_permutations=200 is a reasonable compromise between precision and runtime.
score_observed, perm_scores, p_value = permutation_test_score(
    clf, X, y, cv=cv, n_permutations=200, n_jobs=1, random_state=42,
    scoring="accuracy",
)

print(f"Observed accuracy: {score_observed:.1%}")
print(f"p-value:           {p_value:.4f}")
print(f"Permutation mean:  {perm_scores.mean():.1%} ± {perm_scores.std():.1%}")

In [ ]:
# Visualise the null distribution.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(perm_scores, bins=25, color="lightgrey", edgecolor="grey", label="Null distribution")
ax.axvline(score_observed, color="#e74c3c", linewidth=2, label=f"Observed ({score_observed:.1%})")
ax.axvline(0.5, color="black", linewidth=1, linestyle="--", label="Theoretical chance")

ax.set_xlabel("Accuracy")
ax.set_ylabel("Count")
ax.set_title(f"Permutation test — p = {p_value:.4f}")
ax.legend()
plt.tight_layout()
plt.show()

### Interpretation

Two things to notice:

1. **The null distribution is centred near 50%, but not exactly at 50%.** Individual permutations fluctuate — some produce 55%, others 45%. This is normal. The *width* of the null distribution depends on the sample size: fewer trials → wider distribution → harder to achieve significance.

2. **The p-value tells you whether your result could be a fluke.** If p < 0.05, the answer is: it is unlikely that chance alone produced this accuracy. Note that a significant p-value does not tell you the accuracy is *high enough to be useful* — only that it is *real*. A statistically significant 55% accuracy is real but useless for a BCI.

📖 **Rule of thumb.** Always report a permutation test alongside accuracy. An accuracy without a significance test is an anecdote, not a result.

---

❓ **Exercise.** What would happen to the width of the null distribution if you had 200 trials per class instead of ~45? Would it become easier or harder to reach significance with the same true accuracy? Reason through this before testing.

---

## 3. Scenario B — The auditor

### The situation

A colleague shows you their motor-imagery classifier: 95% accuracy on 64-channel data with 90 trials. You are impressed — until you look at their code and notice something suspicious.

> *"Wait — did you fit the CSP on all the data before splitting into train and test? That would mean the test set influenced the spatial filters."*

This is **data leakage** — the most common and most dangerous methodological error in BCI classification. It inflates accuracy by allowing information from the test set to contaminate the training procedure.

### ❓ Pause — your prediction

1. Why does fitting CSP on the entire dataset (before splitting) cause a problem? What information leaks?
2. Would the leakage be obvious — i.e., would the accuracy look "too good to be true"?
3. In notebook #2, the pipeline was `Pipeline([("CSP", csp), ("LDA", lda)])` inside `cross_val_score`. Does `Pipeline` prevent leakage? Why?

---

### Demonstration — leaky vs correct evaluation

We deliberately introduce leakage by fitting CSP on the full dataset before cross-validation, and compare the result to the correct pipeline.

In [ ]:
# ── CORRECT: CSP is inside the Pipeline, refitted on each training fold. ──
clf_correct = Pipeline([
    ("CSP", CSP(n_components=4, reg=None, log=True, norm_trace=False)),
    ("LDA", LinearDiscriminantAnalysis()),
])
scores_correct = cross_val_score(clf_correct, X, y, cv=cv)
print(f"Correct pipeline:  {scores_correct.mean():.1%} ± {scores_correct.std():.1%}")

In [ ]:
# ── LEAKY: CSP fitted on ALL data, then LDA cross-validated on CSP features. ──
csp_leaky = CSP(n_components=4, reg=None, log=True, norm_trace=False)
csp_leaky.fit(X, y)                      # ← THIS IS THE LEAK
X_leaked = csp_leaky.transform(X)         # features computed from all data

lda = LinearDiscriminantAnalysis()
scores_leaked = cross_val_score(lda, X_leaked, y, cv=cv)
print(f"Leaky pipeline:    {scores_leaked.mean():.1%} ± {scores_leaked.std():.1%}")

In [ ]:
# Side-by-side comparison.
fig, ax = plt.subplots(figsize=(7, 4))
positions = [0, 1]
bp = ax.boxplot([scores_correct, scores_leaked], positions=positions,
                widths=0.5, patch_artist=True)
bp["boxes"][0].set_facecolor("#2980b9")
bp["boxes"][1].set_facecolor("#e74c3c")
ax.set_xticks(positions)
ax.set_xticklabels([f"Correct\n{scores_correct.mean():.1%}",
                     f"Leaky\n{scores_leaked.mean():.1%}"])
ax.set_ylabel("Accuracy")
ax.set_title("Effect of data leakage on reported accuracy")
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()

### Interpretation

The leaky pipeline produces inflated accuracy. The inflation can range from a few percentage points to dramatic overestimation, depending on the dataset size and the number of CSP components.

**Why does this happen?** CSP finds spatial filters that maximise the variance ratio between two classes. If CSP sees the *test* trials during fitting, it optimises filters that separate *those specific trials* — including their noise. The classifier then benefits from filters that are unrealistically well-tuned to the test data.

**How to prevent it:** Any transformation that uses the labels `y` — CSP, xDAWN, feature selection, standardisation fitted on the full dataset — must be inside the cross-validation loop. Scikit-learn's `Pipeline` ensures this automatically: every step in the pipeline is re-fitted on the training fold only.

📖 **The rule is absolute.** If a processing step looks at the class labels, it must be inside the cross-validation loop. No exceptions. No "it probably doesn't matter." Leakage is undetectable from the accuracy number alone — the only safeguard is correct code structure.

---

❓ **Exercise.** A common but subtler form of leakage occurs when you select channels or frequency bands by inspecting the data *before* classification. For example: "I plotted the ERD topography, noticed that C3 and C4 show the strongest effect, and then used only those channels for classification." Why is this a form of leakage if the same data is used for both selection and evaluation?

---

## 4. Scenario C — The debugger

### The situation

You run the MI pipeline on a new subject and obtain 54% accuracy — barely above chance. The permutation test confirms it is not significant (p > 0.05). Your supervisor asks:

> *"Before concluding that this subject cannot do motor imagery, have you checked whether the problem is in your preprocessing, your features, or your classifier?"*

Low accuracy has many possible causes. Blaming the subject (or the paradigm) is the last step, not the first.

### ❓ Pause — your prediction

Before seeing the diagnostic procedure, list at least four things that could cause low accuracy in an MI-BCI pipeline. For each, describe how you would test whether it is the culprit.

---

### A systematic diagnostic procedure

When accuracy is low, work through these checks in order. Each eliminates one category of problem.

| Check | What you test | How |
|---|---|---|
| 1. Signal presence | Is there any ERD at all? | Plot the time-frequency map at C3/C4 |
| 2. Frequency band | Am I looking at the right frequencies? | Compare accuracy across bands |
| 3. Data quantity | Do I have enough trials? | Plot the learning curve |
| 4. Overfitting | Is the model memorising training noise? | Compare train vs test accuracy |

We apply all four checks to subject 1 (where accuracy is reasonable) to establish what "healthy" diagnostics look like, so you can recognise when something is wrong.

### Check 1 — Is there a signal?

Before touching the classifier, verify that the expected neural phenomenon (ERD in the μ band, 8–13 Hz, over C3/C4) is present in the data.

In [ ]:
from mne.time_frequency import tfr_morlet

# Wider epochs for TF analysis.
raw_fnames = eegbci.load_data(1, [4, 8, 12], update_path=True)
raw_tf = concatenate_raws([read_raw_edf(f, preload=True) for f in raw_fnames])
eegbci.standardize(raw_tf)
raw_tf.set_montage(make_standard_montage("standard_1005"))
raw_tf.filter(l_freq=1.0, h_freq=40.0)

events_tf, event_id_tf = mne.events_from_annotations(raw_tf)
event_id_lr = {k: v for k, v in event_id_tf.items() if v in (2, 3)}

epochs_tf = mne.Epochs(raw_tf, events_tf, event_id_lr,
                       tmin=-1.0, tmax=4.0, baseline=None,
                       picks="eeg", preload=True)

freqs = np.arange(4, 30, 1)
n_cycles = freqs / 2.0
tfr = tfr_morlet(epochs_tf, freqs=freqs, n_cycles=n_cycles,
                 return_itc=False, average=True)
tfr.apply_baseline(baseline=(-1.0, 0), mode="percent")

fig = tfr.plot(["C3"], title="TFR at C3 — all MI trials, subject 1", combine="mean")
plt.show()

**What to look for:** A region of decreased power (cool colours) in the 8–13 Hz band during the imagery interval (roughly 0.5–3.5 s). If this is absent, no classifier can extract the signal — the problem is upstream of the pipeline (subject compliance, electrode placement, or genuine BCI illiteracy).

### Check 2 — Is the frequency band correct?

The standard MI pipeline uses 7–30 Hz (μ + β). But the optimal band varies across individuals. If a subject's mu rhythm peaks at 11 Hz rather than 10 Hz, or if their ERD is primarily in the β band, the default 7–30 Hz may dilute the signal with noise from uninformative frequencies.

In [ ]:
# Compare accuracy across different frequency bands.
bands = {
    "μ only (8–13 Hz)":    (8, 13),
    "β only (13–30 Hz)":   (13, 30),
    "μ + β (7–30 Hz)":     (7, 30),
    "Broad (4–40 Hz)":     (4, 40),
}

raw_fnames = eegbci.load_data(1, [4, 8, 12], update_path=True)
raw_base = concatenate_raws([read_raw_edf(f, preload=True) for f in raw_fnames])
eegbci.standardize(raw_base)
raw_base.set_montage(make_standard_montage("standard_1005"))

events_base, eid_base = mne.events_from_annotations(raw_base)
eid_lr = {k: v for k, v in eid_base.items() if v in (2, 3)}

band_results = {}
for name, (fmin, fmax) in bands.items():
    raw_b = raw_base.copy().filter(l_freq=fmin, h_freq=fmax)
    ep_b = mne.Epochs(raw_b, events_base, eid_lr,
                      tmin=0.5, tmax=3.5, baseline=None,
                      picks="eeg", preload=True)
    X_b = ep_b.get_data(copy=False) * 1e6
    y_b = ep_b.events[:, -1]
    sc = cross_val_score(clf, X_b, y_b, cv=cv)
    band_results[name] = sc
    print(f"{name:25s}: {sc.mean():.1%} ± {sc.std():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bp = ax.boxplot(band_results.values(), labels=band_results.keys(),
                patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("#2980b9")
    patch.set_alpha(0.6)
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8)
ax.set_ylabel("Accuracy")
ax.set_title("Classification accuracy by frequency band — subject 1")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

**What to look for:** If one band is substantially better than others, the subject's discriminative signal is concentrated there. If the broadest band performs *worst*, uninformative frequencies are adding noise. If all bands perform near chance, the problem is not the frequency band — move to the next check.

### Check 3 — Do I have enough data?

A learning curve plots accuracy as a function of the number of training examples. It answers two questions: (a) would more data improve performance? (b) is the model underfitting or overfitting?

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    clf, X, y,
    train_sizes=np.linspace(0.2, 1.0, 6),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy",
    n_jobs=1,
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), "o-", color="#e74c3c", label="Training accuracy")
ax.fill_between(train_sizes,
                train_scores.mean(axis=1) - train_scores.std(axis=1),
                train_scores.mean(axis=1) + train_scores.std(axis=1),
                alpha=0.15, color="#e74c3c")
ax.plot(train_sizes, test_scores.mean(axis=1), "o-", color="#2980b9", label="Test accuracy")
ax.fill_between(train_sizes,
                test_scores.mean(axis=1) - test_scores.std(axis=1),
                test_scores.mean(axis=1) + test_scores.std(axis=1),
                alpha=0.15, color="#2980b9")
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8)
ax.set_xlabel("Number of training trials")
ax.set_ylabel("Accuracy")
ax.set_title("Learning curve — CSP + LDA, subject 1")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### How to read a learning curve

| Pattern | Diagnosis | Action |
|---|---|---|
| Training ≈ 100%, test much lower | **Overfitting.** The model memorises training noise. | Add regularisation, reduce features, collect more data. |
| Both curves plateau at a low value | **Underfitting.** The model is too simple or the features are uninformative. | Try richer features, different spatial filter, or different classifier. |
| Test curve still rising at the largest training size | **Insufficient data.** More trials would likely help. | Collect more data, or use data augmentation. |
| Training and test curves converge at a high value | **Healthy.** The model generalises well. | The pipeline is working as expected. |

### Check 4 — Train vs test gap (overfitting)

The learning curve already reveals this, but a more direct check is to compare training accuracy to cross-validated test accuracy.

In [ ]:
# Train on all data and evaluate on the same data (training accuracy).
clf_check = Pipeline([
    ("CSP", CSP(n_components=4, reg=None, log=True, norm_trace=False)),
    ("LDA", LinearDiscriminantAnalysis()),
])
clf_check.fit(X, y)
train_acc = clf_check.score(X, y)
test_acc = scores_baseline.mean()

print(f"Training accuracy:        {train_acc:.1%}")
print(f"Cross-validated accuracy:  {test_acc:.1%}")
print(f"Gap:                      {train_acc - test_acc:.1%}")
print()
if train_acc - test_acc > 0.15:
    print("⚠ Large train-test gap — likely overfitting.")
elif train_acc - test_acc < 0.05:
    print("✓ Small train-test gap — model generalises well.")
else:
    print("Moderate gap — some overfitting, but within acceptable range for small datasets.")

---

❓ **Exercise.** Increase the number of CSP components from 4 to 20 and replot the learning curve. What happens to the train-test gap? Why does using more components increase overfitting when the dataset is small?

---

## 5. Scenario D — The realist

### The situation

Your pipeline works well for subject 1. Your supervisor says:

> *"One subject is an anecdote. Run it on ten subjects and show me the distribution."*

This is where BCI research meets reality. The inter-subject variability in motor-imagery performance is large, and a substantial minority of users (estimates range from 15% to 30%) cannot achieve reliable BCI control — a phenomenon known as **BCI illiteracy** or **BCI inefficiency**.

### ❓ Pause — your prediction

1. If you run the pipeline on 10 subjects, what distribution of accuracies would you expect? Uniform? Normal? Skewed?
2. How would you define a "BCI-illiterate" subject — what accuracy threshold marks the boundary?
3. If a subject scores 52%, is it safe to conclude they cannot perform motor imagery? What alternative explanations exist?

---

In [ ]:
# Run the pipeline on subjects 1–10.
# This takes 1–3 minutes depending on download speed and CPU.
n_subjects = 10
subject_scores = {}

for subj in range(1, n_subjects + 1):
    try:
        X_s, y_s, _ = load_mi_subject(subject=subj)
        sc = cross_val_score(clf, X_s, y_s, cv=cv)
        subject_scores[subj] = sc
        print(f"Subject {subj:2d}: {sc.mean():.1%} ± {sc.std():.1%}")
    except Exception as e:
        print(f"Subject {subj:2d}: failed ({e})")

In [ ]:
# Visualise the distribution.
means = [subject_scores[s].mean() for s in sorted(subject_scores)]
stds = [subject_scores[s].std() for s in sorted(subject_scores)]
subjects = sorted(subject_scores)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#e74c3c" if m < 0.6 else "#2980b9" for m in means]
ax.bar(range(len(subjects)), means, yerr=stds, color=colors,
       edgecolor="white", capsize=4, alpha=0.8)
ax.axhline(0.5, color="grey", linestyle="--", linewidth=1, label="Chance (50%)")
ax.axhline(0.7, color="black", linestyle=":", linewidth=1, label="Usability threshold (~70%)")
ax.set_xticks(range(len(subjects)))
ax.set_xticklabels([f"S{s}" for s in subjects])
ax.set_xlabel("Subject")
ax.set_ylabel("Accuracy")
ax.set_title("Motor imagery classification accuracy across subjects")
ax.legend()
ax.set_ylim(0.3, 1.0)
plt.tight_layout()
plt.show()

below_70 = sum(1 for m in means if m < 0.7)
below_60 = sum(1 for m in means if m < 0.6)
print(f"\nSubjects below 70% accuracy: {below_70}/{len(means)}")
print(f"Subjects below 60% accuracy: {below_60}/{len(means)}")

### Interpretation — the BCI illiteracy problem

The distribution across subjects reveals several important realities:

1. **Variability is large.** Accuracies typically range from near-chance to above 90%. This is not a failure of the pipeline — it reflects genuine differences in how strongly individuals modulate their sensorimotor rhythms during imagery.

2. **Some subjects fail.** Those scoring below ~60% (red bars) cannot be distinguished from chance with this pipeline and this amount of data. These subjects may be BCI-illiterate for motor imagery, or they may need a different paradigm, more training, or a different pipeline.

3. **Reporting only the best subject is misleading.** A BCI system must be evaluated across a representative sample. The mean accuracy ± standard deviation across subjects is the primary metric in BCI literature.

4. **Low accuracy ≠ no signal.** Before labelling a subject as BCI-illiterate, run the diagnostic checks from Scenario C (signal presence, frequency band, learning curve). Some "illiterate" subjects respond to a different frequency band or require more training sessions.

📖 **For the BCI field, this is a central challenge.** A system that works for 70% of users is not a universal assistive technology. Current research directions include adaptive algorithms (that adjust to the user over time), transfer learning (using data from other subjects to bootstrap a new user's classifier), and alternative paradigms (P300, SSVEP) for users who cannot modulate motor rhythms.

---

❓ **Exercise.** For the subject with the lowest accuracy, run the full diagnostic from Scenario C: check the TFR at C3 for ERD, compare frequency bands, and plot the learning curve. Is the poor performance due to a pipeline problem or a genuine absence of the signal?

---

## 6. Synthesis — a diagnostic checklist

When evaluating any BCI classification result, work through this checklist:

### Before trusting the number

- **Permutation test.** Is the accuracy statistically significant? (Scenario A)
- **Leakage audit.** Is every label-dependent transformation inside the cross-validation loop? (Scenario B)

### When accuracy is low

- **Signal check.** Is the expected neural phenomenon visible in the time-frequency map? (Scenario C, Check 1)
- **Band check.** Is the frequency band matched to this subject's physiology? (Scenario C, Check 2)
- **Data check.** Does the learning curve suggest more data would help? (Scenario C, Check 3)
- **Overfitting check.** Is there a large gap between training and test accuracy? (Scenario C, Check 4)

### Before generalising

- **Multi-subject evaluation.** Does the pipeline work across subjects, or only for a cherry-picked individual? (Scenario D)
- **Report the distribution**, not just the mean. The variance across subjects is as informative as the average.

---

## 7. Practice scenarios

### Scenario P1

> A colleague reports 98% accuracy on a 2-class motor imagery task with 30 trials per class. They used CSP with 20 components and a random forest with 500 trees. What is your first concern?

<details>
<summary>Click to reveal the analysis</summary>

**Overfitting.** With only 30 trials per class and 20 CSP components, the feature space (20 dimensions) is comparable to the sample size (60 trials). A random forest with 500 trees can easily memorise the training set. The 98% figure almost certainly reflects training accuracy, not genuine generalisation. Ask to see the learning curve and the train-test gap. Also check for data leakage: were the CSP components fitted inside or outside the cross-validation loop?
</details>

### Scenario P2

> You are building a BCI for a locked-in patient. After 3 sessions of motor imagery, accuracy is 58% — not significant by permutation test. The clinical team asks: should you switch to a P300 speller?

<details>
<summary>Click to reveal the analysis</summary>

**Not yet.** First, run the diagnostic procedure: check the TFR for ERD, try different frequency bands, and examine the learning curve. Some users require 5–10 sessions before reliable ERD develops — motor imagery is a learned skill, and early sessions may reflect poor strategy rather than BCI illiteracy. If the diagnostics show no ERD after 5+ sessions, switching to a P300 or SSVEP paradigm is justified — these paradigms rely on passive responses to external stimuli and do not require the user to learn a voluntary modulation strategy.
</details>

### Scenario P3

> Your pipeline achieves 75% on subject 1, but only 51% on subject 2. Both have the same number of trials. The TFR for subject 2 shows clear ERD at C3 during left-hand imagery, but no ERD at C4 during right-hand imagery. What might explain this?

<details>
<summary>Click to reveal the analysis</summary>

**Asymmetric ERD.** Subject 2 produces a clear ERD for one condition but not the other. This means the two classes differ in *one* hemisphere only, reducing the contrast that CSP can exploit. Possible causes: the subject may have a dominant motor strategy that does not differentiate well between left and right, or there may be a neurological asymmetry. Possible solutions: try using a broader frequency range (the β-band lateralisation may be more discriminative), increase the CSP regularisation parameter, or use foot imagery as the second class instead of right hand (since foot imagery produces ERD at Cz, which is more spatially distinct from the C3 focus of left-hand imagery).
</details>

### Scenario P4

> A paper reports "82% average accuracy across 20 subjects" for a novel MI classifier. The supplementary material reveals that hyperparameters (number of CSP components, regularisation strength, classifier type) were selected *per subject* by grid search on the full dataset, then the best configuration was evaluated with cross-validation. Is this valid?

<details>
<summary>Click to reveal the analysis</summary>

**No — this is data leakage.** Hyperparameter selection on the full dataset means the "best" configuration was chosen based on information that includes the test folds. The correct procedure is **nested cross-validation**: an outer loop for evaluation and an inner loop for hyperparameter selection. The inner loop searches over configurations using only the training data from the outer fold; the outer fold's test data is never touched during selection. The reported 82% is likely inflated.
</details>

---

## 8. Key takeaways

1. **An accuracy number without a significance test is an anecdote.** Always run a permutation test. Small datasets can produce impressive-looking accuracies by chance.

2. **Data leakage is undetectable from the result.** The only safeguard is correct code structure. Every label-dependent step must be inside the cross-validation loop. Use `Pipeline`.

3. **Low accuracy is a symptom, not a diagnosis.** Work through the diagnostic checklist — signal presence, frequency band, data quantity, overfitting — before concluding that the problem is unsolvable.

4. **One subject is an anecdote.** BCI results must be reported across a representative sample. The distribution matters as much as the mean.

5. **BCI illiteracy is real but overdiagnosed.** Many "illiterate" subjects respond to a different frequency band, need more training, or would succeed with a different paradigm.

---

## 9. What comes next

- **Notebook #3 — Stimulus-Evoked BCIs.** A second BCI paradigm (P300) that serves users who cannot control oscillatory BCIs.
- **Notebook #3.5 — From Research Question to Analysis Strategy.** Having now seen two BCI paradigms and the evaluation pitfalls, the student is equipped to reason about *which* analysis strategy matches which research question.